In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d vbookshelf/respiratory-sound-database \
       -p /content/icbhi_dataset/raw --unzip --quiet

print("Download complete.")

In [ ]:
import os
import shutil
import numpy as np
import pandas as pd
import librosa                        # audio loading, analysis
import librosa.display                # waveform/spectrogram plotting
import soundfile as sf                # reliable .wav read/write
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from collections import Counter
from IPython.display import Audio, display

In [ ]:
CONFIG = {
    "raw_dir" : "/content/icbhi_dataset/raw/Respiratory_Sound_Database/Respiratory_Sound_Database",
    "cycles_dir" : "/content/icbhi_dataset/cycles",
    "splits_file": "/content/icbhi_dataset/raw/Respiratory_Sound_Database/ICBHI_challenge_train_test.txt",
    "spectrograms_dir": "/content/icbhi_dataset/spectrograms",
    "target_sr" : 22050,
    "class_map"  : {
        (0, 0): "Normal",
        (1, 0): "Crackle",
        (0, 1): "Wheeze",
        (1, 1): "Both",
    },
    "n_fft"       : 1024,
    "hop_length"  : 512,
    "n_mels"      : 128,
    "fmin"        : 50,
    "fmax"        : 2000,
    "img_size"    : 224,
    "classes"    : ["Normal", "Crackle", "Wheeze", "Both"],
    "seed"       : 42,
    "colormap"   : "viridis",
}

In [ ]:
for d in [CONFIG["raw_dir"], CONFIG["cycles_dir"]]:
    os.makedirs(d, exist_ok=True)

print("CONFIG ready.")
print(f"Target sample rate : {CONFIG['target_sr']} Hz")
print(f"Classes            : {CONFIG['classes']}")

In [ ]:
def parse_annotation_file(txt_path: str) -> pd.DataFrame:
  df = pd.read_csv(
        txt_path,
        sep = "\t",
        header=None,
        names = ["start", "end", "crackle", "wheeze"]
                 )
  """
    Read one ICBHI annotation .txt file and return a DataFrame.

    Args:
        txt_path: path to the .txt annotation file

    Returns:
        DataFrame with columns:
            start    → cycle start time in seconds (float)
            end      → cycle end time in seconds (float)
            crackle  → 1 if crackles present, else 0 (int)
            wheeze   → 1 if wheeze present, else 0 (int)
            label    → human-readable class string ("Normal", "Crackle", etc.)
    """
    # Convert crackle/wheeze to int (sometimes read as float)
  df["crackle"] = df["crackle"].astype(int)
  df["wheeze"]  = df["wheeze"].astype(int)

  # Map (crackle, wheeze) turple -> class string using CONFIG
  df["label"] = df.apply(
      lambda row: CONFIG["class_map"][(row["crackle"], row["wheeze"])],
      axis=1
  )
  return df


In [ ]:
def demo_annotation_parsing(raw_dir: str):
    """Show the first annotation file we find."""
    audio_dir = os.path.join(raw_dir, "audio_and_txt_files")

    # Find any .txt file
    txt_files = list(Path(audio_dir).glob("*.txt"))
    if not txt_files:
        print("No annotation files found. Check your raw_dir path.")
        return

    sample_txt = str(txt_files[0])
    print(f"Parsing: {os.path.basename(sample_txt)}\n")

    df = parse_annotation_file(sample_txt)
    print(df.to_string(index=False))
    print(f"\nCycles in this recording: {len(df)}")
    print(f"Label counts: {df['label'].value_counts().to_dict()}")


demo_annotation_parsing(CONFIG["raw_dir"])

In [ ]:
!wget --no-check-certificate -P /content/icbhi_dataset/raw/Respiratory_Sound_Database "https://bhichallenge.med.auth.gr/sites/default/files/ICBHI_final_database/ICBHI_challenge_train_test.txt"

In [ ]:
def build_master_dataframe(raw_dir: str, splits_file: str) -> pd.DataFrame:
    """
    Parse all annotation files and combine into one master DataFrame.

    Also attaches:
        - patient_id  → extracted from filename (first number, e.g. "101")
        - split       → "train" or "test" from official ICBHI split file
        - wav_path    → full path to the source .wav file
        - txt_path    → full path to the annotation .txt file

    Args:
        raw_dir     : path to the raw dataset root
        splits_file : path to ICBHI_challenge_train_test.txt

    Returns:
        master DataFrame with one row per breathing cycle
    """
    audio_dir = os.path.join(raw_dir, "audio_and_txt_files")


    # The split file has one filename per line, labeled "train" or "test"
    # Format:  101_1b1_Al_sc_Litt3200   train
    split_df = pd.read_csv(
        splits_file,
        sep="\t",
        header=None,
        names=["filename", "split"]
    )
    # Build a dict: filename_stem → "train" or "test"
    split_dict = dict(zip(split_df["filename"], split_df["split"]))

    all_rows = []   # collect rows from every file

    txt_files = sorted(Path(audio_dir).glob("*.txt"))
    print(f"Found {len(txt_files)} annotation files...")

    for txt_path in txt_files:
        stem = txt_path.stem               # e.g. "101_1b1_Al_sc_Litt3200"
        wav_path = txt_path.with_suffix(".wav")  # matching .wav file

        if not wav_path.exists():
            # Skip if audio file missing (shouldn't happen in clean download)
            print(f"  WARNING: No .wav found for {stem}, skipping.")
            continue

        # Extract patient ID — always the first part before the first underscore
        patient_id = stem.split("_")[0]    # "101_1b1_..." → "101"

        # Get train/test label from the split dict
        split = split_dict.get(stem, "unknown")

        # Parse this file's annotation
        df = parse_annotation_file(str(txt_path))

        # Attach metadata columns to every row
        df["filename"]   = stem
        df["patient_id"] = patient_id
        df["split"]      = split
        df["wav_path"]   = str(wav_path)
        df["txt_path"]   = str(txt_path)

        all_rows.append(df)

    # Stack all individual DataFrames into one master table
    master = pd.concat(all_rows, ignore_index=True)

    # Add a unique cycle_id for each row (useful for saving files later)
    master["cycle_id"] = master.index

    print(f"\nMaster DataFrame built.")
    print(f"Total breathing cycles : {len(master)}")
    print(f"Unique patients        : {master['patient_id'].nunique()}")
    print(f"Train cycles           : {(master['split'] == 'train').sum()}")
    print(f"Test cycles            : {(master['split'] == 'test').sum()}")

    return master
master_df = build_master_dataframe(CONFIG["raw_dir"], CONFIG["splits_file"])

In [ ]:
def plot_class_distribution(master_df: pd.DataFrame):
    """
    Plot class counts for train and test splits side by side.
    This is your first look at the imbalance problem.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("ICBHI 2017 — Class Distribution per Split", fontsize=14, fontweight="bold")

    colors = {
        "Normal" : "#4CAF50",   # green — healthy
        "Crackle": "#FF9800",   # orange — abnormal
        "Wheeze" : "#F44336",   # red — abnormal
        "Both"   : "#9C27B0",   # purple — most severe
    }

    for ax, split in zip(axes, ["train", "test"]):
        split_df = master_df[master_df["split"] == split]
        counts   = split_df["label"].value_counts()

        # Reorder to match CONFIG["classes"] order
        counts = counts.reindex(CONFIG["classes"], fill_value=0)

        bars = ax.bar(
            counts.index,
            counts.values,
            color=[colors[c] for c in counts.index],
            edgecolor="white",
            linewidth=1.2
        )

        # Annotate each bar with its count and percentage
        total = counts.sum()
        for bar, count in zip(bars, counts.values):
            pct = 100 * count / total
            ax.text(
                bar.get_x() + bar.get_width() / 2,  # x: center of bar
                bar.get_height() + 30,               # y: just above bar
                f"{count}\n({pct:.1f}%)",            # label text
                ha="center", va="bottom", fontsize=9
            )

        ax.set_title(f"{split.capitalize()} Split  (n={total})", fontsize=12)
        ax.set_ylabel("Number of Cycles")
        ax.set_ylim(0, counts.max() * 1.25)   # headroom for annotations
        ax.spines[["top", "right"]].set_visible(False)

    plt.tight_layout()
    plt.savefig("class_distribution.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("\nKey observation: Normal >> Crackle >> Wheeze >> Both")
    print("This imbalance is WHY WeightedRandomSampler + SpecAugment is needed in Phase 4.")

plot_class_distribution(master_df)

In [ ]:
path = "/content/icbhi_dataset/raw/Respiratory_Sound_Database/ICBHI_challenge_train_test.txt"

with open(path) as f:
    lines = f.readlines()

print(f"Total lines: {len(lines)}")
print("First 5 lines:")
for line in lines[:5]:
    print(repr(line))

In [ ]:
def demonstrate_audio_loading(wav_path: str, target_sr: int = 22050):
    """
    Load one .wav file and explain what librosa gives you.

    Args:
        wav_path  : path to any .wav file in the dataset
        target_sr : the sample rate to resample to
    """

    y, sr = librosa.load(
        wav_path,
        sr=target_sr,   # resample to 22050 Hz regardless of original rate
        mono=True
    )
    # y  → shape: (num_samples,) — the actual audio signal

    duration = len(y) / sr   # total duration in seconds


    print(f"File          : {os.path.basename(wav_path)}")
    print(f"Sample rate   : {sr} Hz")
    print(f"Num samples   : {len(y):,}")
    print(f"Duration      : {duration:.2f} seconds")
    print(f"y dtype       : {y.dtype}")
    print(f"y min/max     : {y.min():.4f} / {y.max():.4f}")
    print(f"y shape       : {y.shape}")
    print("\nIntuition:")
    print(f"  {sr} samples/sec × {duration:.1f} sec = {len(y):,} numbers")
    print(f"  Each number = air pressure at that instant (range: -1.0 to +1.0)")


    print("\nPlaying audio in Colab...")
    display(Audio(y, rate=sr))

    return y, sr

# Get a sample audio file path from the master_df
sample_wav_path = master_df["wav_path"].iloc[0]

demonstrate_audio_loading(sample_wav_path, 22050)

In [ ]:
# BREATHING CYCLE SEGMENTATION
# The ICBHI labels are per CYCLE, not per full recording.
# A full recording might be 20 seconds with 8 breathing cycles.
# We need to slice each cycle into its own audio clip.

def segment_and_save_cycles(master_df: pd.DataFrame, cycles_dir: str, target_sr: int):
  """
    For every row in master_df (= every breathing cycle):
        1. Load the source .wav
        2. Slice the audio between start_time and end_time
        3. Save as a new .wav file in cycles_dir/split/label/

    This creates the folder structure needed for ImageFolder later
    (after converts these to spectrograms).

    Args:
        master_df  : the master DataFrame from build_master_dataframe()
        cycles_dir : where to save segmented cycle .wav files
        target_sr  : sample rate to save at
    """

  # Cache loaded audio files to avoid re-reading the same .wav repeatedly.
  # One recording can have 8+ cycles — no need to reload 8 times.
  audio_cache = {}
  saved = 0
  skipped = 0

  for idx, row in master_df.iterrows():
    out_dir = os.path.join(cycles_dir, row["split"], row["label"])
    os.makedirs(out_dir, exist_ok=True)

    cycle_filename = f"{row['filename']}_cycle{row['cycle_id']:04d}.wav"
    out_path = os.path.join(out_dir, cycle_filename)

    if os.path.exists(out_path):
      skipped += 1
      continue

    wav_path = row["wav_path"]
    if wav_path not in audio_cache:
          y, sr = librosa.load(wav_path, sr=target_sr, mono=True)
          audio_cache[wav_path] = (y, sr)
    else:
        y, sr = audio_cache[wav_path]

    start_sample = int(row["start"] * sr)
    end_sample   = int(row["end"]   * sr)

    cycle_audio = y[start_sample:end_sample]


    min_samples = int(0.5 * sr)   # 0.5 seconds × 22050 = 11025 samples
    if len(cycle_audio) < min_samples:
          skipped += 1
          continue

    sf.write(out_path, cycle_audio, sr)   # soundfile handles wav encoding
    saved += 1

  print(f"\nSegmentation complete.")
  print(f"Saved   : {saved} cycles")
  print(f"Skipped : {skipped} cycles (already exist or too short)")

  # Show what the folder structure looks like
  print("\nFolder structure created:")
  for split in ["train", "test"]:
      for label in CONFIG["classes"]:
          folder = os.path.join(cycles_dir, split, label)
          if os.path.exists(folder):
              n = len(os.listdir(folder))
              print(f"  {split}/{label:<8} → {n} cycles")



segment_and_save_cycles(master_df=master_df, cycles_dir=CONFIG["cycles_dir"], target_sr=CONFIG["target_sr"])

In [ ]:
def plot_waveforms_by_class(cycles_dir: str, target_sr: int, num_examples: int = 2):
    """
    Plot sample waveforms for each class side by side.
    This is your intuition-building moment for the audio domain.

    Args:
        cycles_dir   : path to segmented cycles folder
        target_sr    : sample rate
        num_examples : how many examples to show per class
    """
    fig, axes = plt.subplots(
        len(CONFIG["classes"]), num_examples,
        figsize=(14, 4 * len(CONFIG["classes"]))
    )
    fig.suptitle(
        "Lung Sound Waveforms — What Each Class Looks Like in Time Domain",
        fontsize=13, fontweight="bold", y=1.01
    )

    class_descriptions = {
        "Normal" : "Smooth, quiet, rhythmic oscillations",
        "Crackle": "Sharp explosive spikes — sudden airway openings",
        "Wheeze" : "Sustained oscillation — narrowed airway vibration",
        "Both"   : "Combination of spikes and sustained oscillation",
    }

    np.random.seed(CONFIG["seed"])

    for row_idx, label in enumerate(CONFIG["classes"]):
        # Find available cycle files for this class (train split)
        class_folder = os.path.join(cycles_dir, "train", label)
        if not os.path.exists(class_folder):
            print(f"  No samples found for {label} yet. Run segmentation first.")
            continue

        wav_files = list(Path(class_folder).glob("*.wav"))
        if not wav_files:
            continue

        # Pick random examples
        chosen = np.random.choice(
            wav_files,
            size=min(num_examples, len(wav_files)),
            replace=False
        )

        for col_idx, wav_file in enumerate(chosen):
            ax = axes[row_idx][col_idx]

            # Load this cycle
            y, sr = librosa.load(str(wav_file), sr=target_sr)

            # Time axis in seconds
            time_axis = np.linspace(0, len(y) / sr, num=len(y))

            # Plot the waveform
            ax.plot(
                time_axis, y,
                linewidth=0.5,
                color=["#4CAF50","#FF9800","#F44336","#9C27B0"][row_idx],
                alpha=0.8
            )
            ax.set_xlim(0, len(y) / sr)
            ax.set_ylim(-1.0, 1.0)   # amplitude always in [-1, 1] after librosa

            # Labels
            if col_idx == 0:
                ax.set_ylabel(f"{label}\n{class_descriptions[label]}", fontsize=9)
            if row_idx == len(CONFIG["classes"]) - 1:
                ax.set_xlabel("Time (seconds)")

            ax.set_title(f"Example {col_idx + 1}", fontsize=9)
            ax.spines[["top", "right"]].set_visible(False)

    plt.tight_layout()
    plt.savefig("waveforms_by_class.png", dpi=150, bbox_inches="tight")
    plt.show()



plot_waveforms_by_class(cycles_dir=CONFIG["cycles_dir"], target_sr=CONFIG["target_sr"])

In [ ]:
def play_class_examples(cycles_dir: str, target_sr: int):
    """
    Play one audio example per class directly in Colab.
    Run this cell and listen carefully to each sound.
    """
    np.random.seed(CONFIG["seed"])

    for label in CONFIG["classes"]:
        class_folder = os.path.join(cycles_dir, "train", label)
        wav_files = list(Path(class_folder).glob("*.wav"))

        if not wav_files:
            print(f"{label}: no files found")
            continue

        chosen = np.random.choice(wav_files)
        y, sr = librosa.load(str(chosen), sr=target_sr)

        duration = len(y) / sr
        print(f"\n{'='*40}")
        print(f"Class    : {label}")
        print(f"File     : {os.path.basename(chosen)}")
        print(f"Duration : {duration:.2f}s")
        print(f"Listen   ↓")
        display(Audio(y, rate=sr))


play_class_examples(cycles_dir=CONFIG["cycles_dir"], target_sr=CONFIG["target_sr"])

In [ ]:
for split in ["train", "test"]:
  for label in CONFIG["classes"]:
    os.makedirs(
        os.path.join(CONFIG["spectrograms_dir"], split, label),
        exist_ok=True
    )


print("CONFIG ready.")
print(f"Mel parameters: n_fft={CONFIG['n_fft']}, hop={CONFIG['hop_length']}, "
      f"n_mels={CONFIG['n_mels']}, fmin={CONFIG['fmin']}, fmax={CONFIG['fmax']}")

In [ ]:
def explain_parameters():

    sr         = CONFIG["target_sr"]
    n_fft      = CONFIG["n_fft"]
    hop_length = CONFIG["hop_length"]
    n_mels     = CONFIG["n_mels"]

    window_duration_ms = (n_fft / sr) * 1000
    hop_duration_ms    = (hop_length / sr) * 1000
    overlap_pct        = (1 - hop_length / n_fft) * 100
    freq_resolution    = sr / n_fft


    print("  MEL-SPECTROGRAM PARAMETER EXPLANATION")


    print(f"\nn_fft = {n_fft}")
    print(f"  → FFT window size in SAMPLES")
    print(f"  → Duration: {window_duration_ms:.1f} ms per analysis window")
    print(f"  → Frequency resolution: {freq_resolution:.1f} Hz per bin")
    print(f"  → Larger n_fft = better frequency resolution, worse time resolution")
    print(f"  → 1024 is a good balance for respiratory audio (~46ms windows)")

    print(f"\nhop_length = {hop_length}")
    print(f"  → How far to slide the window each step (in SAMPLES)")
    print(f"  → Duration: {hop_duration_ms:.1f} ms per step")
    print(f"  → Overlap: {overlap_pct:.0f}% between consecutive windows")
    print(f"  → Smaller hop = more time frames = wider image (more detail)")
    print(f"  → 512 = 50% overlap, standard for audio analysis")

    print(f"\nn_mels = {n_mels}")
    print(f"  → Number of mel filter banks = IMAGE HEIGHT")
    print(f"  → More mel bins = higher frequency resolution in image")
    print(f"  → 128 is standard; medical papers use 64–256")

    print(f"\nfmin = {CONFIG['fmin']} Hz, fmax = {CONFIG['fmax']} Hz")
    print(f"  → Frequency range to capture in the spectrogram")
    print(f"  → Lung sounds: 50 Hz (low rumbles) to 2000 Hz (high wheezes)")
    print(f"  → Cutting off at 2000 Hz removes irrelevant high-freq noise")

    print(f"\nImage output: {CONFIG['img_size']} × {CONFIG['img_size']} pixels")
    print(f"  → Resized to match ResNet-50 expected input (224×224)")
    print(f"  → 3-channel RGB (colormap converts grayscale → color)")



explain_parameters()

In [ ]:
def compare_representations(wav_path: str):
    """
    Plot STFT spectrogram, Mel-Spectrogram, and MFCC
    for the same audio clip
    """
    # Load audio
    y, sr = librosa.load(wav_path, sr=CONFIG["target_sr"])

    fig = plt.figure(figsize=(18, 10))
    fig.suptitle(
        f"Three Audio Representations — {Path(wav_path).stem}",
        fontsize=13, fontweight="bold"
    )

    # Waveform (baseline reference)
    ax_wave = fig.add_subplot(4, 1, 1)
    time_axis = np.linspace(0, len(y) / sr, num=len(y))
    ax_wave.plot(time_axis, y, color="#2196F3", linewidth=0.5)
    ax_wave.set_title("Waveform (time domain — what Phase 1 showed you)")
    ax_wave.set_ylabel("Amplitude")
    ax_wave.set_xlabel("Time (s)")
    ax_wave.spines[["top", "right"]].set_visible(False)

    # STFT Spectrogram
    # librosa.stft() → complex matrix → take magnitude → convert to dB
    ax_stft = fig.add_subplot(4, 1, 2)

    D = librosa.stft(y, n_fft=CONFIG["n_fft"], hop_length=CONFIG["hop_length"])
    # D shape: (n_fft//2 + 1, time_frames) — complex valued
    # |D| gives us magnitude (energy at each freq × time point)

    D_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)
    # amplitude_to_db: converts to log scale (decibels)
    # Log scale is essential — audio energy spans many orders of magnitude
    # Without log scale, quiet sounds are invisible next to loud ones

    librosa.display.specshow(
        D_db,
        sr=sr,
        hop_length=CONFIG["hop_length"],
        x_axis="time",
        y_axis="hz",        # linear frequency axis
        ax=ax_stft,
        cmap=CONFIG["colormap"]
    )
    ax_stft.set_title("STFT Spectrogram (linear frequency — notice empty top half)")
    ax_stft.set_ylabel("Frequency (Hz)")
    plt.colorbar(ax_stft.collections[0], ax=ax_stft, format="%+2.0f dB")

    # Mel-Spectrogram
    ax_mel = fig.add_subplot(4, 1, 3)

    S = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_fft=CONFIG["n_fft"],
        hop_length=CONFIG["hop_length"],
        n_mels=CONFIG["n_mels"],
        fmin=CONFIG["fmin"],
        fmax=CONFIG["fmax"]
    )
    # S shape: (n_mels, time_frames) — power values (amplitude²)
    # n_mels=128 means 128 frequency bands (after mel warping)

    S_db = librosa.power_to_db(S, ref=np.max)
    # power_to_db: 10 * log10(S / ref)
    # ref=np.max means top of colorbar = 0 dB, everything else is negative

    librosa.display.specshow(
        S_db,
        sr=sr,
        hop_length=CONFIG["hop_length"],
        x_axis="time",
        y_axis="mel",       # mel-warped frequency axis
        fmin=CONFIG["fmin"],
        fmax=CONFIG["fmax"],
        ax=ax_mel,
        cmap=CONFIG["colormap"]
    )
    ax_mel.set_title("Mel-Spectrogram (mel frequency — expanded low-freq region, this is what we'll train on)")
    ax_mel.set_ylabel("Mel Frequency")
    plt.colorbar(ax_mel.collections[0], ax=ax_mel, format="%+2.0f dB")

    # MFCC
    ax_mfcc = fig.add_subplot(4, 1, 4)

    mfccs = librosa.feature.mfcc(
        y=y,
        sr=sr,
        n_mfcc=13,          # 13 coefficients is the classic speech recognition default
        n_fft=CONFIG["n_fft"],
        hop_length=CONFIG["hop_length"]
    )
    # mfccs shape: (13, time_frames)
    # MFCCs are a compressed "fingerprint" of spectral shape
    # They throw away absolute energy, keeping only the shape
    # Useful for speech — less ideal for medical audio where energy matters

    librosa.display.specshow(
        mfccs,
        sr=sr,
        hop_length=CONFIG["hop_length"],
        x_axis="time",
        ax=ax_mfcc,
        cmap="coolwarm"
    )
    ax_mfcc.set_title("MFCC (13 coefficients — compact but discards energy info, not used for training)")
    ax_mfcc.set_ylabel("MFCC Coefficient")
    plt.colorbar(ax_mfcc.collections[0], ax=ax_mfcc)

    plt.tight_layout()
    plt.savefig("three_representations.png", dpi=150, bbox_inches="tight")
    plt.show()

    print("\nKey observations:")
    print("  STFT   → upper half of plot is empty (lung sounds < 2000 Hz)")
    print("  Mel    → full image used efficiently (mel warping fills the space)")
    print("  MFCC   → most compressed, but loses energy information we need")
    print("  Winner → Mel-Spectrogram ✓")

compare_representations(sample_wav_path)

In [ ]:
def plot_mel_spectrograms_by_class(cycles_dir: str, num_examples: int = 3):
    """
    Plot mel-spectrograms for examples of each class.
    This is the core visual intuition for why CNNs can learn these patterns.

    Args:
        cycles_dir   : path to segmented cycles (Phase 1 output)
        num_examples : examples per class
    """
    np.random.seed(CONFIG["seed"])

    n_classes = len(CONFIG["classes"])
    fig, axes = plt.subplots(
        n_classes, num_examples,
        figsize=(5 * num_examples, 4 * n_classes)
    )
    fig.suptitle(
        "Mel-Spectrograms by Class — What CNN Will Learn",
        fontsize=13, fontweight="bold", y=1.01
    )

    # What to look for in each class
    visual_cues = {
        "Normal" : "Smooth texture, low energy, no distinct patterns",
        "Crackle": "Vertical bright streaks = broadband bursts at instants in time",
        "Wheeze" : "Horizontal bright band = sustained tone at one frequency",
        "Both"   : "Vertical streaks AND horizontal bands together",
    }

    colors = {
        "Normal" : "#4CAF50",
        "Crackle": "#FF9800",
        "Wheeze" : "#F44336",
        "Both"   : "#9C27B0"
    }

    for row_idx, label in enumerate(CONFIG["classes"]):
        class_folder = os.path.join(cycles_dir, "train", label)
        wav_files    = list(Path(class_folder).glob("*.wav"))

        if not wav_files:
            print(f"No files for {label} — run Phase 1 segmentation first.")
            continue

        chosen = np.random.choice(
            wav_files,
            size=min(num_examples, len(wav_files)),
            replace=False
        )

        for col_idx, wav_file in enumerate(chosen):
            ax = axes[row_idx][col_idx]

            y, sr = librosa.load(str(wav_file), sr=CONFIG["target_sr"])

            # Compute mel-spectrogram
            S = librosa.feature.melspectrogram(
                y=y, sr=sr,
                n_fft=CONFIG["n_fft"],
                hop_length=CONFIG["hop_length"],
                n_mels=CONFIG["n_mels"],
                fmin=CONFIG["fmin"],
                fmax=CONFIG["fmax"]
            )                           # shape: (128, time_frames)
            S_db = librosa.power_to_db(S, ref=np.max)   # convert to dB scale

            # Display as image
            img = librosa.display.specshow(
                S_db,
                sr=sr,
                hop_length=CONFIG["hop_length"],
                x_axis="time",
                y_axis="mel",
                fmin=CONFIG["fmin"],
                fmax=CONFIG["fmax"],
                ax=ax,
                cmap=CONFIG["colormap"]
            )

            duration = len(y) / sr
            ax.set_title(f"Example {col_idx+1}  ({duration:.1f}s)", fontsize=9)

            if col_idx == 0:
                ax.set_ylabel(
                    f"{label}\n{visual_cues[label]}",
                    fontsize=8,
                    color=colors[label]
                )
            else:
                ax.set_ylabel("")

            # Remove x labels except bottom row
            if row_idx < n_classes - 1:
                ax.set_xlabel("")

    plt.tight_layout()
    plt.savefig("mel_spectrograms_by_class.png", dpi=150, bbox_inches="tight")
    plt.show()


plot_mel_spectrograms_by_class(cycles_dir=CONFIG["cycles_dir"], num_examples=3)

In [ ]:
def wav_to_mel_spectrogram_image(
    wav_path   : str,
    img_size   : int = 224,
    target_sr  : int = 22050,
    n_fft      : int = 1024,
    hop_length : int = 512,
    n_mels     : int = 128,
    fmin       : float = 50.0,
    fmax       : float = 2000.0,
    colormap   : str = "viridis",
) -> np.ndarray:
    y, sr = librosa.load(
        wav_path,
        sr=target_sr,
        mono=True,
    )

    # Pad or trim to fixed length
    # Problem: breathing cycles have variable durations (0.5s to 6+ seconds).
    # CNNs need fixed-size input → we need fixed-size spectrograms.
    # Solution: pad short cycles with zeros, trim long ones.
    # Target: 3 seconds (median breathing cycle duration in ICBHI)

    target_length = 3 * target_sr

    if len(y) < target_length:
      pad_width = target_length - len(y)
      y = np.pad(y, (0, pad_width), mode="constant", constant_values=0)

    else:
      start = (len(y) - target_length) // 2
      y = y[start : start + target_length]

    S = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_fft=n_fft,
        hop_length=hop_length,
        n_mels=n_mels,
        fmin=fmin,
        fmax=fmax
    )

    S_db = librosa.power_to_db(S, ref=np.max)


    S_min, S_max = S_db.min(), S_db.max()

    if S_max > S_min:
        S_norm = (S_db - S_min) / (S_max - S_min)   # normalize to [0, 1]
    else:
        S_norm = np.zeros_like(S_db)

    S_uint8 = (S_norm * 255).astype(np.uint8)

    import matplotlib.pyplot as plt
    from PIL import Image
    cmap = plt.colormaps[colormap]
    S_rgb = cmap(S_norm)
    S_rgb = (S_rgb[:, :, :3] * 255).astype(np.uint8)

    img = Image.fromarray(S_rgb)          # convert numpy → PIL Image
    img = img.resize(
        (img_size, img_size),             # resize to 224×224
        Image.LANCZOS                     # high-quality downsampling filter
    )
    img_array = np.array(img)
    # img_array shape: (224, 224, 3) → final CNN input

    return img_array

In [ ]:
def visualize_conversion_pipeline(wav_path: str):
    from PIL import Image
    import matplotlib.pyplot as plt

    y, sr = librosa.load(wav_path, sr=CONFIG["target_sr"], mono=True)

    # Replicate pipeline steps
    target_length = 3 * sr
    if len(y) < target_length:
        y_fixed = np.pad(y, (0, target_length - len(y)), mode="constant")
    else:
        start = (len(y) - target_length) // 2
        y_fixed = y[start : start + target_length]

    S      = librosa.feature.melspectrogram(
        y=y_fixed, sr=sr,
        n_fft=CONFIG["n_fft"], hop_length=CONFIG["hop_length"],
        n_mels=CONFIG["n_mels"], fmin=CONFIG["fmin"], fmax=CONFIG["fmax"]
    )
    S_db   = librosa.power_to_db(S, ref=np.max)
    S_norm = (S_db - S_db.min()) / (S_db.max() - S_db.min() + 1e-8)

    cmap    = plt.colormaps[CONFIG["colormap"]]
    S_rgb   = (cmap(S_norm)[:, :, :3] * 255).astype(np.uint8)
    S_final = np.array(Image.fromarray(S_rgb).resize(
        (CONFIG["img_size"], CONFIG["img_size"]), Image.LANCZOS
    ))

    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    fig.suptitle("wav_to_mel_spectrogram_image() — Step by Step", fontsize=12)

    # Step 1: Waveform
    axes[0].plot(np.linspace(0, 3, len(y_fixed)), y_fixed, lw=0.4, color="#2196F3")
    axes[0].set_title("Step 1\nWaveform (padded/trimmed to 3s)")
    axes[0].set_xlabel("Time (s)")
    axes[0].set_ylabel("Amplitude")
    axes[0].spines[["top", "right"]].set_visible(False)

    # Step 2: Raw mel-spectrogram (power)
    axes[1].imshow(S, aspect="auto", origin="lower", cmap="gray")
    axes[1].set_title(f"Step 2\nMel-Spectrogram (power)\nshape: {S.shape}")
    axes[1].set_xlabel("Time frames")
    axes[1].set_ylabel("Mel bins")

    # Step 3: Log scale (dB)
    axes[2].imshow(S_db, aspect="auto", origin="lower", cmap="gray")
    axes[2].set_title(f"Step 3\nLog Scale (dB)\nFaint features now visible")
    axes[2].set_xlabel("Time frames")

    # Step 4: Colormap applied
    axes[3].imshow(S_rgb, aspect="auto", origin="lower")
    axes[3].set_title(f"Step 4\nColormap '{CONFIG['colormap']}'\nGrayscale → RGB")
    axes[3].set_xlabel("Time frames")

    # Step 5: Final resized image
    axes[4].imshow(S_final)
    axes[4].set_title(f"Step 5\nFinal: {CONFIG['img_size']}×{CONFIG['img_size']} RGB\nCNN input ✓")
    axes[4].set_xlabel("Pixels")

    plt.tight_layout()
    plt.savefig("conversion_pipeline.png", dpi=150, bbox_inches="tight")
    plt.show()

    print(f"\nInput : .wav file ({len(y)/sr:.2f}s, {len(y):,} samples)")
    print(f"Output: numpy array {S_final.shape}, dtype={S_final.dtype}")
    print(f"This array IS the image the CNN will train on.")

In [ ]:
visualize_conversion_pipeline(sample_wav_path)

In [ ]:
import os
from collections import defaultdict
from pathlib import Path
from tqdm.notebook import tqdm
from PIL import Image # Image is used to save the spectrograms

def convert_all_cycles_to_spectrograms(cycles_dir: str, spectrograms_dir: str):
    stats = defaultdict(int)   # track counts per class

    for split in ["train", "test"]:
        for label in CONFIG["classes"]:

            in_folder  = os.path.join(cycles_dir, split, label)
            out_folder = os.path.join(spectrograms_dir, split, label)
            os.makedirs(out_folder, exist_ok=True)

            wav_files = list(Path(in_folder).glob("*.wav"))

            if not wav_files:
                print(f"  No .wav files in {split}/{label} — skipping.")
                continue

            print(f"\nConverting {split}/{label} ({len(wav_files)} files)...")

            for wav_file in tqdm(wav_files, desc=f"{split}/{label}", unit="file"):

                # Output PNG path (same stem, different extension)
                out_path = os.path.join(
                    out_folder,
                    wav_file.stem + ".png"
                )

                # Skip if already converted (resume-safe — same pattern as Phase 1)
                if os.path.exists(out_path):
                    stats[f"{split}/{label}_skipped"] += 1
                    continue

                try:
                    # Convert .wav → 224×224 RGB numpy array
                    img_array = wav_to_mel_spectrogram_image(
                        str(wav_file),
                        img_size   = CONFIG["img_size"],
                        target_sr  = CONFIG["target_sr"],
                        n_fft      = CONFIG["n_fft"],
                        hop_length = CONFIG["hop_length"],
                        n_mels     = CONFIG["n_mels"],
                        fmin       = CONFIG["fmin"],
                        fmax       = CONFIG["fmax"],
                        colormap   = CONFIG["colormap"],
                    )

                    # Save as PNG using PIL
                    Image.fromarray(img_array).save(out_path)
                    stats[f"{split}/{label}"] += 1

                except Exception as e:
                    print(f"  ERROR converting {wav_file.name}: {e}")
                    stats["errors"] += 1



    print("  CONVERSION COMPLETE")
    print("=" * 50)
    total = 0
    for split in ["train", "test"]:
        print(f"\n  {split.upper()}")
        for label in CONFIG["classes"]:
            key = f"{split}/{label}"
            n   = stats.get(key, 0)
            total += n
            print(f"    {label:<10}: {n:>5} PNGs")
    print(f"\n  Total saved : {total:,}")
    print(f"  Errors      : {stats.get('errors', 0)}")

In [ ]:
os.makedirs(CONFIG["spectrograms_dir"], exist_ok=True)
convert_all_cycles_to_spectrograms(
    cycles_dir=CONFIG["cycles_dir"],
    spectrograms_dir=CONFIG["spectrograms_dir"]
)

In [ ]:
import os
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision
from torchvision import transforms, models
from PIL import Image
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, auc, roc_auc_score
)
from sklearn.preprocessing import label_binarize

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


In [ ]:
CONFIG = {
    "raw_dir" : "/content/icbhi_dataset/raw/Respiratory_Sound_Database/Respiratory_Sound_Database",
    "cycles_dir" : "/content/icbhi_dataset/cycles",
    "splits_file": "/content/icbhi_dataset/raw/Respiratory_Sound_Database/ICBHI_challenge_train_test.txt",
    "spectrograms_dir": "/content/icbhi_dataset/spectrograms",
    "checkpoint_path"  : "/content/checkpoints/resnet50_respiratory.pt",
    "best_model_path"  : "/content/checkpoints/resnet50_respiratory_best.pt",
    "target_sr" : 22050,
    "class_map"  : {
        (0, 0): "Normal",
        (1, 0): "Crackle",
        (0, 1): "Wheeze",
        (1, 1): "Both",
    },

    "class_counts_train": {
        "Normal" : 2031,
        "Crackle": 1209,
        "Wheeze" : 499,
        "Both"   : 363,
    },

    "n_fft"       : 1024,
    "hop_length"  : 512,
    "n_mels"      : 128,
    "fmin"        : 50,
    "fmax"        : 2000,
    "img_size"    : 224,
    "classes"     : ["Both", "Crackle", "Normal", "Wheeze"],
    "num_classes" : 4,
    "seed"       : 42,
    "colormap"   : "viridis",

    "norm_mean"   : [0.485, 0.456, 0.406],
    "norm_std"    : [0.229, 0.224, 0.225],

    "spec_augment_prob"     : 0.5,
    "freq_mask_max_width"   : 24,
    "time_mask_max_width"   : 24,
    "num_freq_masks"        : 2,
    "num_time_masks"        : 2,

    "batch_size"        : 32,
    "num_epochs"         : 30,
    "learning_rate"       : 1e-4,
    "weight_decay"        : 1e-4,
    "label_smoothing"     : 0.1,
    "early_stopping_patience" : 7,
    "num_workers"         : 2,

}

In [ ]:
os.makedirs(os.path.dirname(CONFIG["checkpoint_path"]), exist_ok=True)
torch.manual_seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])

print(f"Device: {device}")
print(f"Classes (alphabetical, matches ImageFolder): {CONFIG['classes']}")

**SPECAUGMENT: IMPLEMENTED FROM SCRATCH**




```
SpecAugment (Park et al. 2019, originally for speech recognition) is the
standard augmentation technique for spectrogram-based audio classification.

The idea: randomly mask out horizontal or vertical strips of the spectrogram
during training, forcing the model to not over-rely on any single
frequency band or time window.

  FREQUENCY MASKING: blank out a horizontal band (a range of frequencies)
    → Forces model to not rely on one specific frequency band
    → Simulates: stethoscope picking up muffled frequencies, partial
      occlusion of certain harmonics

  TIME MASKING: blank out a vertical band (a range of time steps)
    → Forces model to not rely on one specific moment in the breathing cycle
    → Simulates: brief recording dropouts, patient movement artifacts

This operates directly on the spectrogram IMAGE (after it's loaded as a
tensor), not on the raw audio. That's what makes it different from
standard image augmentation (rotation, flip) — those make no physical
sense for spectrograms (flipping a spectrogram vertically would reverse
the frequency axis, which is physically meaningless).
```



In [ ]:
class SpecAugment:
    def __init__(
        self, freq_mask_max_width, time_mask_max_width,
         num_freq_masks, num_time_masks, prob
    ):
        self.freq_mask_max_width = freq_mask_max_width
        self.time_mask_max_width = time_mask_max_width
        self.num_freq_masks      = num_freq_masks
        self.num_time_masks      = num_time_masks
        self.prob                = prob



    def __call__(self, spec: torch.Tensor) -> torch.Tensor:

        if np.random.rand() > self.prob:
            return spec

        spec = spec.clone()   # don't mutate the original tensor in-place
        _, H, W = spec.shape  # H = frequency bins, W = time frames

        #Frequency masking (horizontal bands)
        for _ in range(self.num_freq_masks):
            f_width = np.random.randint(0, self.freq_mask_max_width)
            if f_width == 0:
                continue
            f_start = np.random.randint(0, max(1, H - f_width))
            spec[:, f_start:f_start + f_width, :] = 0.0
            # Sets a horizontal strip (range of frequencies, all time steps) to 0

        #Time masking (vertical bands)
        for _ in range(self.num_time_masks):
            t_width = np.random.randint(0, self.time_mask_max_width)
            if t_width == 0:
                continue
            t_start = np.random.randint(0, max(1, W - t_width))
            spec[:, :, t_start:t_start + t_width] = 0.0
            # Sets a vertical strip (range of time, all frequencies) to 0

        return spec

In [ ]:
def visualize_specaugment(dataset, num_examples: int = 4):

    spec_augment = SpecAugment(
        freq_mask_max_width = CONFIG["freq_mask_max_width"],
        time_mask_max_width = CONFIG["time_mask_max_width"],
        num_freq_masks      = CONFIG["num_freq_masks"],
        num_time_masks      = CONFIG["num_time_masks"],
        prob                = 1.0,   # force apply for visualization
    )

    fig, axes = plt.subplots(2, num_examples, figsize=(4 * num_examples, 8))
    fig.suptitle("SpecAugment: Before (top) vs After (bottom)", fontsize=13)

    indices = np.random.choice(len(dataset), num_examples, replace=False)

    for col, idx in enumerate(indices):
        img_tensor, label_idx = dataset[idx]


        mean = torch.tensor(CONFIG["norm_mean"]).view(3, 1, 1)
        std  = torch.tensor(CONFIG["norm_std"]).view(3, 1, 1)
        denorm = img_tensor * std + mean
        denorm = torch.clamp(denorm, 0, 1)

        augmented = spec_augment(denorm)

        axes[0, col].imshow(denorm.permute(1, 2, 0).numpy())
        axes[0, col].set_title(f"{CONFIG['classes'][label_idx]}", fontsize=9)
        axes[0, col].axis("off")

        axes[1, col].imshow(augmented.permute(1, 2, 0).numpy())
        axes[1, col].axis("off")

    plt.tight_layout()
    plt.savefig("specaugment_preview.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
class RespiratorySpectrogramDataset(Dataset):
    """
    Loads spectrogram PNGs from the Phase 2 ImageFolder-style directory.

    We don't use torchvision.datasets.ImageFolder directly because we want
    fine control over WHERE in the pipeline SpecAugment is applied (after
    ToTensor, before Normalize) — ImageFolder's transform argument applies
    everything as one composed transform, making it awkward to inject
    tensor-space augmentation cleanly alongside PIL-space transforms.
    """

    def __init__(self, root_dir: str, classes: list, base_transform,
                 spec_augment=None):
        """
        Args:
            root_dir       : path to train/ or test/ folder
            classes        : CONFIG["classes"] — alphabetically ordered list
            base_transform : standard PIL transforms (resize, ToTensor, Normalize)
            spec_augment   : SpecAugment instance, or None for no augmentation
                             (use None for validation/test sets — never augment eval data)
        """
        self.samples = []   # list of (filepath, label_idx) tuples
        self.classes = classes
        self.class_to_idx = {cls: i for i, cls in enumerate(classes)}
        self.base_transform = base_transform
        self.spec_augment = spec_augment

        for cls in classes:
            cls_dir = os.path.join(root_dir, cls)
            if not os.path.isdir(cls_dir):
                continue
            for fname in os.listdir(cls_dir):
                if fname.endswith(".png"):
                    self.samples.append((
                        os.path.join(cls_dir, fname),
                        self.class_to_idx[cls]
                    ))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        filepath, label = self.samples[idx]

        img = Image.open(filepath).convert("RGB")
        img_tensor = self.base_transform(img)   # resize, ToTensor, Normalize

        if self.spec_augment is not None:
            img_tensor = self.spec_augment(img_tensor)

        return img_tensor, label


def verify_class_index_mapping(dataset: RespiratorySpectrogramDataset):

    print("Class → Index mapping:")
    for cls, idx in dataset.class_to_idx.items():
        print(f"  {idx}: {cls}")

    expected = {cls: i for i, cls in enumerate(CONFIG["classes"])}
    assert dataset.class_to_idx == expected, (
        "MISMATCH: dataset class_to_idx does not match CONFIG['classes'] order!"
    )
    print("\n✓ Verified: class index mapping matches CONFIG['classes']")


In [ ]:
base_transform_train = transforms.Compose([
    transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
    transforms.ToTensor(),
    transforms.Normalize(mean=CONFIG["norm_mean"], std=CONFIG["norm_std"]),
])


base_transform_eval = transforms.Compose([
    transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
    transforms.ToTensor(),
    transforms.Normalize(mean=CONFIG["norm_mean"], std=CONFIG["norm_std"]),
])

In [ ]:
def build_weighted_sampler(dataset: RespiratorySpectrogramDataset) -> WeightedRandomSampler:


    class_sample_counts = np.zeros(CONFIG["num_classes"])
    for _, label in dataset.samples:
        class_sample_counts[label] += 1

    print("Verified class counts in dataset:")
    for cls, idx in dataset.class_to_idx.items():
        print(f"  {cls:<10}: {int(class_sample_counts[idx])}")

    # Weight per CLASS = 1 / count  (rarer class → higher weight)
    class_weights = 1.0 / class_sample_counts

    # Weight per SAMPLE = weight of its class
    sample_weights = np.array([
        class_weights[label] for _, label in dataset.samples
    ])

    sampler = WeightedRandomSampler(
        weights=torch.from_numpy(sample_weights).double(),
        num_samples=len(sample_weights),
        replacement=True

    )

    # Sanity check
    sampled_indices = list(sampler)
    sampled_labels = [dataset.samples[i][1] for i in sampled_indices]
    sampled_counts = np.bincount(sampled_labels, minlength=CONFIG["num_classes"])

    print("\nSimulated epoch sampling distribution (should be roughly equal):")
    for cls, idx in dataset.class_to_idx.items():
        print(f"  {cls:<10}: {sampled_counts[idx]} draws "
              f"({100*sampled_counts[idx]/len(sampled_labels):.1f}%)")

    return sampler


In [ ]:
def build_loss_function():
    return nn.CrossEntropyLoss(label_smoothing=CONFIG["label_smoothing"])

In [ ]:
def build_model(num_classes: int) -> nn.Module:

    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

    # Replace the final fully-connected layer
    # Original: nn.Linear(2048, 1000) for ImageNet's 1000 classes
    # New:      nn.Linear(2048, 4) for our 4 respiratory classes
    in_features = model.fc.in_features   # 2048 for ResNet-50
    model.fc = nn.Sequential(
        nn.Dropout(0.3),

        nn.Linear(in_features, num_classes)
    )

    return model.to(device)




def train_one_epoch(model, loader, criterion, optimizer, device):

    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(loader, desc="Train", leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)              # shape: (batch, 4)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        pbar.set_postfix(loss=loss.item())

    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):

    model.eval()
    running_loss = 0.0
    all_labels = []
    all_preds = []
    all_probs = []   # needed for AUC-ROC — raw softmax probabilities

    for images, labels in tqdm(loader, desc="Eval", leave=False):
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)

        probs = F.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, dim=1)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

    avg_loss = running_loss / len(all_labels)
    accuracy = np.mean(np.array(all_labels) == np.array(all_preds))

    return {
        "loss": avg_loss,
        "accuracy": accuracy,
        "labels": np.array(all_labels),
        "preds": np.array(all_preds),
        "probs": np.array(all_probs),   # shape: (n_samples, 4)
    }


def save_checkpoint(state: dict, path: str):
    torch.save(state, path)


def load_checkpoint(path: str, model, optimizer):

    if not os.path.exists(path):
        print("No checkpoint found — starting fresh.")
        return 0, {"best_val_auc": 0.0, "epochs_without_improvement": 0}

    checkpoint = torch.load(path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    print(f"Resumed from checkpoint at epoch {checkpoint['epoch']}")
    return checkpoint["epoch"], {
        "best_val_auc": checkpoint.get("best_val_auc", 0.0),
        "epochs_without_improvement": checkpoint.get("epochs_without_improvement", 0),
    }


def run_training(model, train_loader, val_loader, criterion, optimizer):

    start_epoch, state = load_checkpoint(CONFIG["checkpoint_path"], model, optimizer)
    best_val_auc = state["best_val_auc"]
    epochs_without_improvement = state["epochs_without_improvement"]

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "val_auc": []}

    for epoch in range(start_epoch, CONFIG["num_epochs"]):
        print(f"\nEpoch {epoch+1}/{CONFIG['num_epochs']}")

        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_results = evaluate(model, val_loader, criterion, device)


        val_labels_binarized = label_binarize(
            val_results["labels"], classes=list(range(CONFIG["num_classes"]))
        )
        val_auc = roc_auc_score(
            val_labels_binarized, val_results["probs"],
            average="macro", multi_class="ovr"
        )

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_results["loss"])
        history["val_acc"].append(val_results["accuracy"])
        history["val_auc"].append(val_auc)

        print(f"  Train loss: {train_loss:.4f}  acc: {train_acc:.4f}")
        print(f"  Val   loss: {val_results['loss']:.4f}  acc: {val_results['accuracy']:.4f}  "
              f"macro-AUC: {val_auc:.4f}")


        is_best = val_auc > best_val_auc
        if is_best:
            best_val_auc = val_auc
            epochs_without_improvement = 0
            torch.save(model.state_dict(), CONFIG["best_model_path"])
            print(f"  ✓ New best model saved (AUC: {best_val_auc:.4f})")
        else:
            epochs_without_improvement += 1

        save_checkpoint({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_auc": best_val_auc,
            "epochs_without_improvement": epochs_without_improvement,
        }, CONFIG["checkpoint_path"])

        # Early stopping
        if epochs_without_improvement >= CONFIG["early_stopping_patience"]:
            print(f"\nEarly stopping triggered — no improvement for "
                  f"{CONFIG['early_stopping_patience']} epochs.")
            break

    return history


def plot_training_history(history: dict):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    axes[0].plot(history["train_loss"], label="Train")
    axes[0].plot(history["val_loss"], label="Val")
    axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()

    axes[1].plot(history["train_acc"], label="Train")
    axes[1].plot(history["val_acc"], label="Val")
    axes[1].set_title("Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()

    axes[2].plot(history["val_auc"], label="Val macro-AUC", color="green")
    axes[2].set_title("Validation Macro AUC-ROC"); axes[2].set_xlabel("Epoch"); axes[2].legend()

    plt.tight_layout()
    plt.savefig("training_history.png", dpi=150, bbox_inches="tight")
    plt.show()



def compute_per_class_sensitivity_specificity(labels, preds, classes):

    results = {}
    cm = confusion_matrix(labels, preds, labels=list(range(len(classes))))

    for idx, cls in enumerate(classes):
        TP = cm[idx, idx]
        FN = cm[idx, :].sum() - TP
        FP = cm[:, idx].sum() - TP
        TN = cm.sum() - TP - FN - FP

        sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        specificity = TN / (TN + FP) if (TN + FP) > 0 else 0.0

        results[cls] = {
            "sensitivity": sensitivity,
            "specificity": specificity,
            "TP": int(TP), "FN": int(FN), "FP": int(FP), "TN": int(TN),
        }

    return results, cm


def plot_confusion_matrix(cm, classes):
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=classes, yticklabels=classes, ax=ax
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title("Confusion Matrix — Test Set")
    plt.tight_layout()
    plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
    plt.show()


def plot_roc_curves(labels, probs, classes):
    """
    Plot one-vs-rest ROC curve for each class, with AUC in the legend.
    """
    labels_bin = label_binarize(labels, classes=list(range(len(classes))))

    fig, ax = plt.subplots(figsize=(7, 6))
    colors = ["#9C27B0", "#FF9800", "#4CAF50", "#F44336"]

    for idx, cls in enumerate(classes):
        fpr, tpr, _ = roc_curve(labels_bin[:, idx], probs[:, idx])
        roc_auc_val = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=colors[idx], label=f"{cls} (AUC = {roc_auc_val:.3f})")

    ax.plot([0, 1], [0, 1], "k--", alpha=0.4, label="Random chance")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC Curves — One-vs-Rest per Class")
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig("roc_curves.png", dpi=150, bbox_inches="tight")
    plt.show()


def youden_index_threshold(labels_binary, probs_for_class):

    fpr, tpr, thresholds = roc_curve(labels_binary, probs_for_class)
    j_scores = tpr - fpr   # Youden's J = TPR - FPR = Sensitivity - (1 - Specificity)
    best_idx = np.argmax(j_scores)

    return thresholds[best_idx], tpr[best_idx], 1 - fpr[best_idx]


def full_clinical_report(test_results: dict, classes: list):

    labels, preds, probs = test_results["labels"], test_results["preds"], test_results["probs"]


    print("  CLINICAL EVALUATION REPORT — TEST SET")


    print(f"\nOverall accuracy: {test_results['accuracy']:.4f}")
    print(f"(Reminder: accuracy alone is misleading on imbalanced data — "
          f"see per-class metrics below)\n")

    metrics, cm = compute_per_class_sensitivity_specificity(labels, preds, classes)

    print(f"{'Class':<10} {'Sensitivity':<12} {'Specificity':<12} {'TP':<6} {'FN':<6} {'FP':<6} {'TN':<6}")

    for cls, m in metrics.items():
        print(f"{cls:<10} {m['sensitivity']:<12.3f} {m['specificity']:<12.3f} "
              f"{m['TP']:<6} {m['FN']:<6} {m['FP']:<6} {m['TN']:<6}")

    print("\nConfusion Matrix ")
    plot_confusion_matrix(cm, classes)

    print("\n ROC Curves ")
    plot_roc_curves(labels, probs, classes)

    print("\n Youden-Optimal Thresholds (vs default 0.5) ")
    labels_bin = label_binarize(labels, classes=list(range(len(classes))))
    for idx, cls in enumerate(classes):
        thresh, sens, spec = youden_index_threshold(labels_bin[:, idx], probs[:, idx])
        print(f"  {cls:<10}: optimal threshold = {thresh:.3f}  "
              f"(sensitivity={sens:.3f}, specificity={spec:.3f})")

    print("\n sklearn classification_report (for reference) ")
    print(classification_report(labels, preds, target_names=classes, digits=3))




In [ ]:
train_dir = os.path.join(CONFIG["spectrograms_dir"], "train")
test_dir  = os.path.join(CONFIG["spectrograms_dir"], "test")

spec_augment = SpecAugment(
    freq_mask_max_width = CONFIG["freq_mask_max_width"],
    time_mask_max_width = CONFIG["time_mask_max_width"],
    num_freq_masks      = CONFIG["num_freq_masks"],
    num_time_masks      = CONFIG["num_time_masks"],
    prob                = CONFIG["spec_augment_prob"],
)


train_dataset = RespiratorySpectrogramDataset(
    train_dir, CONFIG["classes"], base_transform_train, spec_augment=spec_augment
)
test_dataset = RespiratorySpectrogramDataset(
    test_dir, CONFIG["classes"], base_transform_eval, spec_augment=None
    # never augment test data
    )


verify_class_index_mapping(train_dataset)

visualize_specaugment(train_dataset)
sampler = build_weighted_sampler(train_dataset)

train_loader = DataLoader(
    train_dataset, batch_size=CONFIG["batch_size"],
    sampler=sampler,                    # sampler replaces shuffle=True
    num_workers=CONFIG["num_workers"]
)
test_loader = DataLoader(
    test_dataset, batch_size=CONFIG["batch_size"],
    shuffle=False, num_workers=CONFIG["num_workers"]
)

model = build_model(CONFIG["num_classes"])
criterion = build_loss_function()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"]
)


history = run_training(model, train_loader, test_loader, criterion, optimizer)
plot_training_history(history)


# Load best model and run full clinical evaluation
model.load_state_dict(torch.load(CONFIG["best_model_path"]))
test_results = evaluate(model, test_loader, criterion, device)
full_clinical_report(test_results, CONFIG["classes"])